In [ ]:
import pandas as pd

# Path to data frame with WSIs
# df_path = "D:\DATA\with_snomed_category.csv"
# df_path = r"D:\DATA\abmil_exp2.csv"
df_path = r"D:\DATA\abmil_exp3.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to output dir
output_path = r"D:\NOTEBOOKS\Christine\all_slides\feature_summary_new.csv"

# Path to zarr
zarr_dir = r"D:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
all_filenames = df_all["filename"].tolist()
print("Number of files: ", len(all_filenames))

In [ ]:
from helper_functions import with_tissue_artifact

# Path to cache file
cache_file = "cache_tissue_artifact.pkl"

with_tissue = False
if with_tissue: 
    df_tissue = with_tissue_artifact(df_all, cache_file, segmentation_type = "tissue", status="complete", version="default")
    with_tissue = list(set(df_tissue["filename"].tolist()))
    print("WSIs with completed default tissue detection: ", len(with_tissue))
    
    df_sub = df_all[df_all["filename"].isin(with_tissue)].copy()

else: 
    df_sub = df_all.copy()

In [ ]:
from helper_functions import subset_df, subset_df_list

subset = False

if subset:
    df_HE = subset_df(df_sub, "stain", "HE")
    df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
    # df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")
    df_HE = subset_df_list(df_HE, "T_category", "Blood, Bone Marrow and Lymphatic System")
    # df_HE = df_HE[df_HE['M_category'].apply(len) == 1]
else:
    df_HE = df_sub.copy()

In [ ]:
# Feature Results
df_features = pd.read_csv(output_path)
model = 'h-optimus-0'

missing_features = False

if missing_features:
    # Keep only completed + correct model
    df_completed = df_features[(df_features['model'] == model) & (df_features['status'] == "feature extraction complete")].copy()

    with_features = set(df_completed["wsi_path"])
    without_features = df_HE[~df_HE["filename"].isin(with_features)]

    all_filenames = without_features["filename"].tolist()
    print("Files with missing feature extraction: ", len(missing_filenames))

else: 
    all_filenames = df_HE["filename"].tolist()
    print("Number of files: ", len(all_filenames))

In [ ]:
from feature_extraction import ExtractMany

extractor = ExtractMany(all_filenames, output_path, local_zarr_dir = zarr_dir, model = model, remove_artifacts = False)